In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser  

/Users/rahuljauhari/Rahul Jauhari/Personal Projects/GenAI - Learning/Langchain/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/rahuljauhari/Rahul Jauhari/Personal Projects/GenAI - Learning/Langchain/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# Sequential Chain — The Building Block of LangChain

## What is LCEL?

**LCEL** stands for **LangChain Expression Language**. It's the modern, recommended way to build LangChain applications.

The core idea is simple: chain steps together using the `|` (pipe) operator, exactly like in Unix shell:

```
cat file.txt | grep "error" | sort | uniq
```

In LangChain:

```python
chain = prompt | model | output_parser
result = chain.invoke({"technology": "AI"})
```

## What is a Sequential Chain?

A **sequential chain** runs steps one after another. The output of step N becomes the input of step N+1:

```
PromptTemplate → fills in variables → produces a filled prompt string
      ↓
ChatOllama → sends the prompt to the LLM → produces an AIMessage
      ↓
StrOutputParser → extracts the text from AIMessage → produces a plain string
```

## The three building blocks

### 1. PromptTemplate
Turns your template + variables into a filled prompt:
```python
prompt = PromptTemplate(
    template="Tell me 5 facts about {technology}",
    input_variables=["technology"]
)
```

### 2. ChatOllama / ChatOpenAI
Sends the prompt to the LLM and returns an `AIMessage`:
```python
model = ChatOllama(model="llama3.1")
```

### 3. StrOutputParser
Extracts just the text string from the `AIMessage`:
```python
parser = StrOutputParser()
# AIMessage(content="AI is...") → "AI is..."
```

## What you'll learn in this notebook

- How to build a 3-step sequential chain with `|`
- How each piece feeds into the next
- How to run the chain with `.invoke()`
- How to visualise the chain's structure with `.get_graph().print_ascii()`

## Prerequisites

- Ollama running with a model pulled
- Virtual environment activated

In [2]:
prompt = PromptTemplate(
    template="Tell me 5 facts about {technology} technology ",
    input_variables=["technology"],
)

In [ ]:
model = ChatOllama(model="qwen3-coder:30b")

In [4]:
parser = StrOutputParser()

In [5]:
chain = prompt | model | parser

In [6]:
result = chain.invoke({"technology": "AI"})
result

"Here are 5 key facts about AI technology:\n\n1. **AI is already integrated into daily life** - You use AI countless times daily through voice assistants like Siri or Alexa, recommendation systems on Netflix and Spotify, GPS navigation, and even your smartphone's camera features that automatically enhance photos.\n\n2. **AI can process information faster than humans** - Modern AI systems can analyze vast amounts of data in seconds, making them incredibly efficient for tasks like medical diagnosis, financial trading, and pattern recognition that would take humans much longer to complete.\n\n3. **AI has different types and capabilities** - There's narrow AI (designed for specific tasks like playing chess or recognizing faces), general AI (theoretical human-level intelligence), and superintelligence (hypothetical AI that surpasses human intelligence). Currently, we're primarily using narrow AI.\n\n4. **AI is becoming more creative** - Advanced AI systems can now generate original music, w

In [7]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOllama |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


## Chain graph visualisation

`.get_graph().print_ascii()` draws a text diagram of the chain's structure. This is useful for understanding and debugging complex chains — especially when chains start nesting and branching.

Above you can see the flow: `PromptInput → PromptTemplate → ChatOllama → StrOutputParser → Output`.

## Streaming (bonus)

Instead of waiting for the full response, you can stream it word-by-word with `.stream()`:

```python
for chunk in chain.stream({"technology": "blockchain"}):
    print(chunk, end="", flush=True)
```

This gives a much more responsive feel in UI applications.

## What you learned

| Concept | Code |
|---------|------|
| Build a sequential chain | `chain = prompt \| model \| parser` |
| Run the chain | `chain.invoke({"technology": "AI"})` |
| Visualise the structure | `chain.get_graph().print_ascii()` |
| Stream output | `for chunk in chain.stream({...}):` |

**Next:** `ParallelChain.ipynb` — run multiple chains simultaneously on the same input